<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 2–3 Extension Lab: Batches, Partitions, and Profile</h1>
<p>Target Doris 4.1.3 · Independent ext_* lab tables</p>
</div>

[Extension home](README.md) · [Main course contents](../README.md)

Run after completing main Labs 2 and 3. Use the course's single-container sandbox to generate a fixed 100,000-row teaching dataset without mixing it into WWI or business metrics.
Rebuild only `ext_small_batches`, `ext_large_batch`, and `ext_partitioned`; do not rerun the same lab database concurrently in two kernels.
Allow 25–35 minutes, outside the original video duration. Read the SQL before running it; after failure, retain the output and restart from initialization, resetting only these three tables.

Validation: all three tables have identical data; the selected day has 10,000 rows totaling 100,000.00; submit plans, batch/version observations, and Profile observations.
Do not require small batches to create a backlog or claim a speedup based on one timing measurement.

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql, show_records
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. Fixed Input and Table Structure

Each 10,000 rows represents one day, with the amount fixed at 10.00. The first two tables have identical schemas and differ only in write batch sizes.
The third keeps the same key columns and bucketing, adding only daily partitions; this avoids changing sort keys and bucket counts at the same time.

In [ ]:
from time import perf_counter
fields = "order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(12,2) NOT NULL"
partitions = ",".join(f"PARTITION p{day:02} VALUES [('2026-01-{day:02}'), ('2026-01-{day+1:02}'))" for day in range(1,11))
for table in ("ext_small_batches", "ext_large_batch", "ext_partitioned"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    partition = f"PARTITION BY RANGE(order_date) ({partitions})" if table == "ext_partitioned" else ""
    ddl = f'CREATE TABLE {table} ({fields}) DUPLICATE KEY(order_date,order_id) {partition} DISTRIBUTED BY HASH(order_id) BUCKETS 4 PROPERTIES("replication_num"="1")'
    show_sql("Physical design", ddl)
    lab.execute(ddl)
source = """SELECT DATE_ADD(CAST('2026-01-01' AS DATE), INTERVAL CAST(FLOOR(number / 10000) AS INT) DAY),
number, CAST(10 AS DECIMAL(12,2)) FROM numbers("number"="100000")"""
write_measurements = []
previous_group = lab.query("SELECT @@group_commit")[0][0]
try:
    lab.execute("SET group_commit = 'off_mode'")
    for table, batch_size in (("ext_small_batches",1000), ("ext_large_batch",100000), ("ext_partitioned",100000)):
        start = perf_counter()
        for offset in range(0,100000,batch_size):
            lab.execute(f"INSERT INTO {table} {source} WHERE number >= {offset} AND number < {offset+batch_size}")
        write_measurements.append({"Table": table, "Submitted batches": 100000//batch_size,
                                   "Write seconds": round(perf_counter()-start,3)})
        expect(lab.query(f"SELECT COUNT(*), SUM(amount),COUNT(DISTINCT order_id) FROM {table}"), [(100000,"1000000.00",100000)])
        lab.sql(f"SHOW TABLETS FROM {table}", title="Tablet status after writes (not fixed expected values)")
finally:
    lab.execute("SET group_commit = %s", (previous_group,))
show_records("Write measurements", write_measurements)
for table in ("ext_small_batches", "ext_partitioned"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} a FULL OUTER JOIN ext_large_batch b ON a.order_id=b.order_id WHERE a.order_id IS NULL OR b.order_id IS NULL OR a.order_date<>b.order_date OR a.amount<>b.amount"), [(0,)])

## 2. Results, Pruning, and Profile

Run the same predicate on the unpartitioned and partitioned tables. Warm up once, then alternate between them for five rounds, retaining every duration; this is not a concurrent benchmark or P95.
Use `EXPLAIN` to inspect partition/Tablet pruning and Profile to inspect actual rows scanned and operator times. Identical query output does not imply identical scan volume.
You can continue inspecting Rowsets through Course 2's `SHOW TABLET → DetailCmd → CompactionStatus` path; do not trigger manual Compaction.

In [ ]:
predicate = "order_date = '2026-01-03'"
query_measurements = []
previous_profile = lab.query("SELECT @@enable_profile")[0][0]
try:
    lab.execute("SET enable_profile = true")
    for table in ("ext_large_batch", "ext_partitioned"):
        lab.sql(f"EXPLAIN SELECT SUM(amount) FROM {table} WHERE {predicate}", title=table + " plan")
        expect(lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}"), [(10000,"100000.00")])
    for iteration in range(5):
        for table in ("ext_large_batch", "ext_partitioned"):
            start = perf_counter()
            expect(lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}"), [(10000,"100000.00")])
            query_measurements.append({"Iteration": iteration+1, "Table": table,
                                       "Query ms": round((perf_counter()-start)*1000,3)})
    show_records("Query timing comparison", query_measurements)
    lab.sql("SHOW QUERY PROFILE", title="Find this lab's Profile by SQL, database name, and time")
finally:
    lab.execute("SET enable_profile = %s", (previous_profile,))

## 3. Independent Explanation and Troubleshooting

Record the partitions/tablets fields in both plans and find the corresponding scan operators in Profile; if duration does not decrease, explain it using data size, caching, and fixed query overhead without changing results.
Change the small batch to 5,000 rows and rerun the lab, comparing actual batch counts and VersionCount rather than requiring version counts to equal batch counts exactly.
If the Profile list has no record yet, rerun `SHOW QUERY PROFILE` later; for detailed field descriptions, see [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/).
The finally block restores the session setting; retain all three tables for troubleshooting at the end rather than deleting the data.

In [ ]:
lab.close()